# Prova con un guasto finto di ampiezza nota

Tutti i risultati della relazione sono negativi: il residuo dell'autoencoder non
separa le classi. Prima di scriverlo bisogna escludere la spiegazione piu
banale, cioe che sia sbagliata la catena di calcolo.

Il problema di un risultato negativo e che non ha un termine di paragone: se il
residuo non sale sui cuscinetti guasti, non si sa se il metodo non funzioni o
se il codice non misuri quello che dice di misurare. Questo notebook costruisce
il termine di paragone che manca, prendendo cuscinetti **sani** e aggiungendo al
loro segnale un guasto **finto**, di forma e ampiezza decise da noi.

Se il residuo cresce con l'ampiezza di quel guasto, la catena funziona e il
risultato negativo sui guasti veri e un fatto. Se non cresce, il problema e nel
codice.

Il guasto finto e un treno di impulsi smorzati alla frequenza BPFO, che e il
modello classico del difetto localizzato su pista esterna. Non pretende di
riprodurre la fisica del guasto nella corrente di statore: serve come segnale
di prova con una ampiezza nota, non come simulazione.

Accanto al treno di impulsi si prova anche l'aggiunta di **rumore bianco di
pari energia**. Il confronto fra i due dice se il residuo reagisca alla forma
del disturbo o soltanto alla sua energia.

Le prove vengono ripetute su due autoencoder: quello della replica letterale,
che soffre del troncamento della SELU, e quello sul segnale scalato, dove il
troncamento non c'e. Cosi si vede anche se sia il troncamento a nascondere il
guasto iniettato.

Notebook autonomo: non modifica `funzioni.py` e non scrive nulla che gli altri
notebook leggano.

In [ ]:
!apt-get -qq update && apt-get -qq install -y unrar
!pip -q install requests scipy scikit-learn

In [ ]:
import os, sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

REPO = 'https://github.com/matpaol/MacchineEdAzionamentiExam'
possibili = ['.', '..', '../codice',
             '/content/drive/MyDrive/MacchineEdAzionamentiExam',
             '/content/MacchineEdAzionamentiExam']
percorso_codice = None
for c in possibili:
    if os.path.exists(os.path.join(c, 'funzioni.py')):
        percorso_codice = os.path.abspath(c)
        break
if percorso_codice is None:
    subprocess.run(['git', 'clone', '-q', REPO, '/content/MacchineEdAzionamentiExam'],
                   check=True)
    percorso_codice = '/content/MacchineEdAzionamentiExam'
sys.path.insert(0, percorso_codice)
import config
import funzioni as f

f.stile_grafici()
P = config.percorsi(sottocartella='extra_guasto_finto')
dev = f.dispositivo()
seme = 0
print('codice da', percorso_codice, '| dispositivo', dev)

## I dati e i due autoencoder

Gli stessi della replica e del Capitolo 7, con lo stesso seme: diciassette
cuscinetti, un solo regime, 2560 frame sani per l'addestramento.

In [ ]:
cuscinetti = config.CUSCINETTI_PAPER
regime = config.REGIME_PRINCIPALE

f.estrai_misure(cuscinetti, P['raw'], P['estratti'])
inv = f.inventario(cuscinetti, P['estratti'], regimi=[regime]).reset_index(drop=True)
segmenti, anagrafica = f.costruisci_segmenti(inv, segmenti_per_registrazione=4)

lunghezza_frame = f.lunghezza_frame(config.REGIMI[regime]['rpm'])
frame_per_segmento = segmenti.shape[1] // lunghezza_frame
frame = segmenti.reshape(-1, lunghezza_frame)
classe_del_frame = np.repeat(anagrafica['classe'].values, frame_per_segmento)

rng = np.random.default_rng(seme)
sani = np.flatnonzero(classe_del_frame == 0)
scelti = rng.choice(sani, size=2560, replace=False)
rng.shuffle(scelti)
frame_train, frame_val = frame[scelti[:2048]], frame[scelti[2048:]]

# i frame sani NON usati per addestrare: sono quelli su cui si inietta il guasto
mai_visti = np.setdiff1d(sani, scelti)
print(len(frame), 'frame |', len(sani), 'sani | usati per addestrare', len(scelti),
      '| disponibili per la prova', len(mai_visti))

In [ ]:
fattore, massimo = f.scala_globale(frame_train)
print('fattore di scala:', round(fattore, 6))
print()

modelli = {}
print('--- autoencoder della replica letterale, segnale non scalato ---')
modelli['non scalato'] = (f.addestra_dae(frame_train, frame_val, uscita_selu=True,
                                         epoche=500, lotto=256, passo=3e-4,
                                         pazienza=None, seme=seme, dev=dev,
                                         stampa_ogni=250)[0], 1.0)
print()
print('--- autoencoder sul segnale scalato ---')
modelli['scalato'] = (f.addestra_dae(frame_train * fattore, frame_val * fattore,
                                     uscita_selu=True, epoche=500, lotto=256,
                                     passo=3e-4, pazienza=None, seme=seme, dev=dev,
                                     stampa_ogni=250)[0], fattore)

def residuo_medio(nome, blocco):
    """MSE per frame, sempre riportato in ampere al quadrato."""
    modello, k = modelli[nome]
    r = f.calcola_residui(modello, np.asarray(blocco, dtype=np.float32) * k, dev=dev) / k
    return f.mse_per_frame(r)

print()
print('controllo contro il Capitolo 7: residuo dei sani con il modello non scalato',
      round(float(np.mean(residuo_medio('non scalato', frame[classe_del_frame == 0]))), 4),
      '(atteso 0,0801)')

## Il guasto finto

Un difetto localizzato sulla pista esterna produce un impulso ogni volta che un
corpo volvente ci passa sopra, cioe alla frequenza BPFO. Ogni impulso eccita
una risonanza della struttura e si smorza in fretta.

Il segnale di prova riproduce questa forma: una sinusoide a 3~kHz moltiplicata
per un esponenziale decrescente, ripetuta al periodo della BPFO. La fase di
partenza viene sorteggiata per ogni frame, altrimenti l'autoencoder vedrebbe
sempre lo stesso disturbo nella stessa posizione.

L'ampiezza si esprime come frazione del valore efficace del frame a cui viene
sommato, cosi l'unita di misura e la stessa per tutti i frame.

In [ ]:
def treno_impulsi(lunghezza, fs, f_guasto, fase, f_risonanza=3000.0, smorzamento=900.0):
    """Impulsi smorzati ripetuti al periodo del guasto, normalizzati a picco unitario."""
    t = np.arange(lunghezza) / fs
    periodo = 1.0 / f_guasto
    s = np.zeros(lunghezza)
    inizio = fase * periodo
    while inizio < t[-1]:
        dt = t - inizio
        m = dt >= 0
        s[m] += np.exp(-smorzamento * dt[m]) * np.sin(2 * np.pi * f_risonanza * dt[m])
        inizio += periodo
    picco = np.max(np.abs(s))
    return s / picco if picco > 0 else s


rpm = config.REGIMI[regime]['rpm']
frequenze = f.frequenze_guasto(rpm / 60.0)
BPFO = frequenze['BPFO']
print('a {} rpm: BPFO = {:.2f} Hz, cioe {:.2f} impulsi in un giro'.format(
    rpm, BPFO, BPFO / (rpm / 60.0)))

esempio = treno_impulsi(lunghezza_frame, config.FS_ATTESO, BPFO, 0.0)
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.plot(np.arange(lunghezza_frame) / config.FS_ATTESO * 1000, esempio,
        color=f.COLORI['accento'], lw=0.8)
ax.set_xlabel('tempo (ms)')
ax.set_ylabel('ampiezza normalizzata')
ax.set_title('Il guasto finto: un giro di albero, {:.2f} impulsi'.format(
    BPFO / (rpm / 60.0)), fontsize=10)
f.salva_figura(fig, 'guasto_finto_forma', P['figure'])
plt.show()

## La prova

Si prendono 2000 frame sani mai usati per addestrare. A ognuno si somma il
guasto finto con ampiezza crescente, da zero al 50\% del valore efficace del
frame. Per ogni ampiezza si misura il residuo medio e lo si confronta con
quello dei frame sani lasciati intatti.

Le due grandezze riportate sono le stesse usate in tutta la relazione: il
rapporto fra residuo alterato e residuo sano, e l'area sotto la curva fra i due
gruppi.

Accanto al treno di impulsi viene provato il rumore bianco, riscalato in modo
da avere lo stesso valore efficace del treno di impulsi a parita di ampiezza.

In [ ]:
from sklearn.metrics import roc_auc_score

AMPIEZZE = [0.0, 0.02, 0.05, 0.10, 0.20, 0.50, 1.00, 2.00]
N_PROVA = 2000

scelta = rng.choice(mai_visti, size=min(N_PROVA, len(mai_visti)), replace=False)
base = frame[scelta].astype(np.float64)
rms_frame = np.sqrt(np.mean(base ** 2, axis=1, keepdims=True))

# un treno di impulsi per ogni frame, con fase sorteggiata
fasi = rng.uniform(0, 1, size=len(base))
impulsi = np.stack([treno_impulsi(lunghezza_frame, config.FS_ATTESO, BPFO, ph)
                    for ph in fasi])
rumore = rng.normal(size=base.shape)
# stesso valore efficace del treno di impulsi, frame per frame
rumore *= (np.sqrt(np.mean(impulsi ** 2, axis=1, keepdims=True))
           / np.sqrt(np.mean(rumore ** 2, axis=1, keepdims=True)))

righe = []
for nome in modelli:
    riferimento = float(np.mean(residuo_medio(nome, base)))
    for disturbo, etichetta in [(impulsi, 'impulsi a BPFO'), (rumore, 'rumore bianco')]:
        for a in AMPIEZZE:
            alterato = base + a * rms_frame * disturbo
            mse_alt = residuo_medio(nome, alterato)
            mse_san = residuo_medio(nome, base)
            etichette = np.r_[np.zeros(len(mse_san)), np.ones(len(mse_alt))]
            punteggi = np.r_[mse_san, mse_alt]
            righe.append({'modello': nome, 'disturbo': etichetta, 'ampiezza': a,
                          'residuo_sano': riferimento,
                          'residuo_alterato': float(np.mean(mse_alt)),
                          'rapporto': float(np.mean(mse_alt)) / riferimento,
                          'auc': float(roc_auc_score(etichette, punteggi))})

prova = pd.DataFrame(righe)
for nome in modelli:
    for etichetta in prova['disturbo'].unique():
        p = prova[(prova['modello'] == nome) & (prova['disturbo'] == etichetta)]
        print('---', nome, '|', etichetta)
        print(p[['ampiezza', 'residuo_alterato', 'rapporto', 'auc']]
              .round(4).to_string(index=False))
        print()

## Esito della verifica

Il criterio viene dichiarato prima di guardare i numeri, e chiede due cose al
modello sul segnale scalato, cioe quello che non soffre del troncamento.

La prima e la **monotonia**: aumentando l'ampiezza del guasto iniettato, il
residuo e l'area sotto la curva non devono mai scendere. La seconda e che
all'ampiezza massima, dove l'impulso ha un picco pari al doppio del valore
efficace del segnale e quindi un guasto enorme, l'area sotto la curva arrivi
almeno a **0,90**: un autoencoder che ricostruisce il sano e che non se ne
accorge non sta misurando nulla.

Se le due condizioni sono soddisfatte, la catena reagisce a un guasto di
ampiezza nota e i risultati negativi sui guasti veri sono un fatto. Se non lo
sono, il codice non misura quello che dovrebbe e tutta la relazione va rimessa
in discussione.

In [ ]:
SOGLIA_AUC = 0.90

esiti = []
for nome in modelli:
    p = prova[(prova['modello'] == nome) &
              (prova['disturbo'] == 'impulsi a BPFO')].sort_values('ampiezza')
    esiti.append({
        'modello': nome,
        'residuo non scende mai': bool(np.all(np.diff(p['rapporto'].values) > -1e-6)),
        'AUC non scende mai': bool(np.all(np.diff(p['auc'].values) > -1e-6)),
        'rapporto al massimo': float(p['rapporto'].iloc[-1]),
        'AUC al massimo': float(p['auc'].iloc[-1]),
        'AUC oltre la soglia': bool(p['auc'].iloc[-1] >= SOGLIA_AUC)})
esiti = pd.DataFrame(esiti)
print(esiti.round(4).to_string(index=False))
print()

riga = esiti[esiti['modello'] == 'scalato'].iloc[0]
superata = bool(riga['residuo non scende mai'] and riga['AUC non scende mai']
                and riga['AUC oltre la soglia'])
if superata:
    print('VERIFICA SUPERATA. La catena reagisce a un guasto di ampiezza nota:')
    print('  il residuo cresce con l ampiezza e all estremo l AUC vale',
          round(riga['AUC al massimo'], 3), 'contro una soglia di', SOGLIA_AUC)
    print('  quindi i risultati negativi sui guasti veri non sono un errore di codice.')
else:
    print('VERIFICA FALLITA.')
    if not riga['AUC oltre la soglia']:
        print('  all ampiezza massima l AUC vale', round(riga['AUC al massimo'], 3),
              'contro una soglia di', SOGLIA_AUC)
    if not (riga['residuo non scende mai'] and riga['AUC non scende mai']):
        print('  la risposta non e monotona: il residuo non segue l ampiezza')
    print('  prima di usare qualunque risultato negativo bisogna rivedere la catena.')

## Quanto deve essere grande un guasto perche si veda

I cuscinetti guasti veri, nella replica, alzano il residuo del 9,5\% rispetto ai
sani. La cella cerca quale ampiezza del guasto finto produce lo stesso effetto:
e una misura di quanto sia sensibile il metodo, espressa in una unita
interpretabile.

In [ ]:
RAPPORTO_REALE = 1.095

for nome in modelli:
    p = prova[(prova['modello'] == nome) &
              (prova['disturbo'] == 'impulsi a BPFO')].sort_values('ampiezza')
    r, a = p['rapporto'].values, p['ampiezza'].values
    if r[-1] < RAPPORTO_REALE:
        print('{:12s}: nemmeno al {:.0f}% il guasto finto arriva al rapporto dei guasti veri'
              ' (massimo {:.3f})'.format(nome, 100 * a[-1], r[-1]))
    else:
        soglia = float(np.interp(RAPPORTO_REALE, r, a))
        print('{:12s}: serve un impulso al {:.1f}% del valore efficace per ottenere'
              ' il rapporto 1,095 dei guasti veri'.format(nome, 100 * soglia))

## Figura e salvataggi

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(12, 4.4))
stile = {('non scalato', 'impulsi a BPFO'): (f.COLORI['accento'], '-'),
         ('non scalato', 'rumore bianco'): (f.COLORI['accento'], '--'),
         ('scalato', 'impulsi a BPFO'): (f.COLORI['nostro'], '-'),
         ('scalato', 'rumore bianco'): (f.COLORI['nostro'], '--')}

for (nome, etichetta), (colore, tratto) in stile.items():
    p = prova[(prova['modello'] == nome) &
              (prova['disturbo'] == etichetta)].sort_values('ampiezza')
    x = 100 * p['ampiezza']
    assi[0].plot(x, p['rapporto'], tratto, color=colore, marker='o', ms=4,
                 label='{}, {}'.format(nome, etichetta))
    assi[1].plot(x, p['auc'], tratto, color=colore, marker='o', ms=4)

assi[0].axhline(RAPPORTO_REALE, color=f.COLORI['neutro'], ls=':', lw=1)
assi[0].text(0.5, RAPPORTO_REALE, ' guasti veri: 1,095', va='bottom', fontsize=7,
             color=f.COLORI['scuro'])
assi[0].set_xlabel('ampiezza del guasto finto (\% del valore efficace)')
assi[0].set_ylabel('residuo alterato / residuo sano')
assi[0].set_title('Il residuo reagisce al guasto iniettato?', fontsize=10)
assi[0].legend(fontsize=7)

assi[1].axhline(0.5, color=f.COLORI['scuro'], lw=1)
assi[1].set_ylim(0.4, 1.02)
assi[1].set_xlabel('ampiezza del guasto finto (\% del valore efficace)')
assi[1].set_ylabel('AUC fra sano e sano alterato')
assi[1].set_title('La separazione segue l ampiezza?', fontsize=10)

f.salva_figura(fig, 'guasto_finto_risposta', P['figure'])
plt.show()

f.salva_tabella(prova, 'guasto_finto', P['tabelle'])
f.salva_tabella(esiti, 'guasto_finto_esiti', P['tabelle'])